## 파인튜닝에 맞는 데이터 형태 변환

In [1]:
!pip install openai

  Using cached openai-1.64.0-py3-none-any.whl.metadata (27 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.7-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
Using cached openai-1.64.0-py3-none-any.whl (472 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.7-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 74.9 MB/s eta 0:00:00
Using cached h11-0.14.0-py3-none-any.whl (58 kB)


In [2]:
pip install langchain_community langchain_openai faiss-cpu pypdf

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 74.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   --------------------------- ------------ 9.4/13.7 MB 104.5 MB/s eta 0:00:01
   -------------------------------------- - 13.1/13.7 MB 32.2 MB/s eta 0:00:01
   ---------------------------------------- 13.7/13.7 MB 30.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 69.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -------------------- ------------------- 6.6/12.6 MB 37.0 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 31.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 83.8 MB/s eta 0:00:00
   -------------------------

In [ ]:
import json
import os
from collections import defaultdict
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.schema import Document

# JSON 파일 로드
#json_file_path = "./data_preprocessed/information_reclassified_부정강조0221.json"
json_file_path = "./sample.json"

with open(json_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# FAISS 인덱스 로드
faiss_passage_all_path = "./faiss_index/faiss_index_passage_all"
faiss_passage_required_path = "./faiss_index/faiss_index_passage_required"

embeddings_model = OpenAIEmbeddings(openai_api_key="api-key")

vector_db_passage_all = FAISS.load_local(faiss_passage_all_path, embeddings_model,allow_dangerous_deserialization=True)

if os.path.exists(faiss_passage_required_path):
    vector_db_passage_required = FAISS.load_local(faiss_passage_required_path, embeddings_model,allow_dangerous_deserialization=True)
else:
    vector_db_passage_required = None


# FAISS 기반 문서 검색 함수
def search_relevant_passages(subjects, topics):
    query = ", ".join([subjects] + topics)  # 검색 키워드 생성
    results = vector_db_passage_all.similarity_search(query, k=3)

    if vector_db_passage_required:
        required_results = vector_db_passage_required.similarity_search(query, k=2)
    else:
        required_results = []

    # 중복 제거 및 최종 검색 결과 추출 (최대 5개)
    unique_results = {res.page_content: res for res in results}
    for doc in required_results:
        unique_results.setdefault(doc.page_content, doc)

    return list(unique_results.values())[:5]  # 최대 5개 반환


# 데이터 변환 함수
def transform_grouped_data(data):
    transformed_data = []

    for entry in data:
        passage = entry["passage"]  
        subject = entry["subject"]
        topics = entry["topics"] 
        question_type = entry["question_type"]

        # RAG 기반 관련 문서 검색
        relevant_passages = search_relevant_passages(subject, topics)
        passage_guidelines = "".join([doc.page_content for doc in relevant_passages])

        question_content = entry["question_text"] + " " + " ".join(
            [f"{choice['number']}. {choice['text']}" for choice in entry["choices"]]
        )

        transformed_entry = {
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "당신은 대한민국 대학수학능력시험 국어영역 독서과목 지문과 문항을 출제하는 한국교육과정평가원 출제위원입니다. "
                        "본 시험은 2015 개정 국어과 교육과정을 바탕으로, 대학에서의 원만하고 능률적인 수학(修學)에 필요한 국어 능력을 측정하는 것을 목표로 합니다. "
                        "고등학교 국어과 교육과정 중 ‘독서’ 과목의 학습 목표와 내용을 중심으로 다양한 소재의 담화 및 글, 자료를 활용하여 학생들의 국어 능력을 측정한다는 목표가 지문 및 문항 구성에도 반영되어야 합니다. "
                        "지문은 인문･예술, 사회･문화, 과학･기술 등 다양한 분야를 아우르며, 설명문･논설문･보고서 등 여러 유형의 글을 활용합니다. "
                        "문항은 사실적･추론적･비판적･창의적 사고 능력을 종합적으로 평가하도록 구성합니다. "
                        "문항은 지문 내용을 정확히 이해했는지를 확인하는 것뿐만 아니라, 지문 속 정보와 개념을 활용하여 논리적으로 추론하거나 새로운 상황에 적용할 수 있는지 평가하는 방향으로 출제해야 합니다. "
                        "객관식 문항은 5지선다형으로 구성하되, 학생들이 피상적인 기억이 아니라 깊이 있는 이해와 사고를 통해 정답을 도출하도록 합니다."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"다음의 분야와 제재를 반영하여 대한민국 수학능력시험 국어영역 독서과목 문제 풀이를 위한 지문을 작성하세요. "
                        f"분야 : {', '.join(subject)}, 제재 : {', '.join(topics)}"
                        f"참고할 문서 내용:{passage_guidelines}"
                    ),
                },
                {"role": "assistant", "content": passage},
                {
                    "role": "user",
                    "content": f"위 지문에 대한 5지선다형 객관식 문제와 선택지를 문제 유형에 맞춰 생성하세요. 문제 유형 : {', '.join(question_type)}"
                    f"참고할 문서 내용:{question_guidelines}",
                },
                {"role": "assistant", "content": question_content},
            ]
        }

        transformed_data.append(transformed_entry)

    return transformed_data


# 데이터 변환
converted_data = transform_grouped_data(data)

# 변환된 데이터 저장
output_file_path = "./프롬프트변경_test.json"
with open(output_file_path, "w", encoding="utf-8") as outfile:
    json.dump(converted_data, outfile, ensure_ascii=False, indent=4)

print(f"변환된 JSON 파일이 저장되었습니다: {output_file_path}")


변환된 JSON 파일이 저장되었습니다: ./프롬프트변경_test.json
